In [2]:
import pandas as pd
import numpy as np
import re

In [3]:
file_path = "final companies .xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="New 20 Unique Prospects"
)

df.head()

,Company Name,Industry,CEO/Founder Name,CEO/Founder Email,Verified / Current Email,Contact Number,Website
0,Staffyn,Recruitment & Staffing,Not Found,Not Found,hr@staffyn.in,+91 88752 84059,https://www.staffyn.in/
1,Progres InnoTech Pvt Ltd,Recruitment & HR Tech / Software,Not Found,Not Found,hr@progresinnotech.com,+91 78299 16411,https://progresinnotech.com/
2,Aarizon Services,Technology Recruitment & Staffing,Not Found,Not Found,contact@aarizon.com,+91 70562 19573,https://www.aarizon.com/
3,Origin Hiring,IT Recruitment & Staffing,Not Found,Not Found,info@originhiring.com,Not Found,https://www.originhiring.com/
4,CipherSchools,EdTech / Career Education,Anurag Mishra — Founder,Not Found,support@cipherschools.com,Not Found,https://www.cipherschools.com/


In [4]:

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 203
Columns: 7


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Company Name              203 non-null    str   
 1   Industry                  203 non-null    str   
 2   CEO/Founder Name          201 non-null    str   
 3   CEO/Founder Email         203 non-null    str   
 4   Verified / Current Email  199 non-null    str   
 5   Contact Number            196 non-null    object
 6   Website                   203 non-null    str   
dtypes: object(1), str(6)
memory usage: 11.2+ KB


In [6]:
df.isna().sum()

Company Name                0
Industry                    0
CEO/Founder Name            2
CEO/Founder Email           0
Verified / Current Email    4
Contact Number              7
Website                     0
dtype: int64

In [7]:
clean_df = df.copy()

print("Original rows:", len(df))
print("Working rows:", len(clean_df))

Original rows: 203
Working rows: 203


In [8]:
clean_df.columns = [
    "Company",
    "Industry",
    "Leadership",
    "CEO_Email",
    "Current_Email",
    "Phone",
    "Website"
]

clean_df.columns

Index(['Company', 'Industry', 'Leadership', 'CEO_Email', 'Current_Email',
       'Phone', 'Website'],
      dtype='str')

In [9]:
for col in clean_df.columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

In [10]:
clean_df.head()

,Company,Industry,Leadership,CEO_Email,Current_Email,Phone,Website
0,Staffyn,Recruitment & Staffing,Not Found,Not Found,hr@staffyn.in,+91 88752 84059,https://www.staffyn.in/
1,Progres InnoTech Pvt Ltd,Recruitment & HR Tech / Software,Not Found,Not Found,hr@progresinnotech.com,+91 78299 16411,https://progresinnotech.com/
2,Aarizon Services,Technology Recruitment & Staffing,Not Found,Not Found,contact@aarizon.com,+91 70562 19573,https://www.aarizon.com/
3,Origin Hiring,IT Recruitment & Staffing,Not Found,Not Found,info@originhiring.com,Not Found,https://www.originhiring.com/
4,CipherSchools,EdTech / Career Education,Anurag Mishra — Founder,Not Found,support@cipherschools.com,Not Found,https://www.cipherschools.com/


In [11]:
missing_values = [
    "",
    " ",
    "Not Found",
    "not found",
    "NOT FOUND",
    "Not published",
    "not published",
    "N/A",
    "n/a",
    "NA",
    "None",
    "-",
    "Contact form only",
    "Contact form only — verify",
    "Contact form only - verify"
]

clean_df = clean_df.replace(
    missing_values,
    np.nan
)

In [12]:
clean_df.isna().sum()

Company            0
Industry           0
Leadership        21
CEO_Email        191
Current_Email     30
Phone             55
Website           10
dtype: int64

In [13]:
print("Number of unique raw industries:",
      clean_df["Industry"].nunique())

Number of unique raw industries: 184


In [14]:
industry_list = (
    clean_df["Industry"]
    .value_counts()
    .reset_index()
)

industry_list.columns = [
    "Industry",
    "Company_Count"
]

industry_list

,Industry,Company_Count
0,InsurTech,5
1,PropTech / Real Estate,4
2,Logistics / Last-Mile Delivery,2
3,Automotive / Used Cars,2
4,Healthcare / Dental Support Org,2
...,...,...
179,B2B Commerce / FinTech,1
180,Fitness / HealthTech,1
181,HealthTech / Fitness,1
182,EdTech / Higher Education,1


In [15]:
industry_list

,Industry,Company_Count
0,InsurTech,5
1,PropTech / Real Estate,4
2,Logistics / Last-Mile Delivery,2
3,Automotive / Used Cars,2
4,Healthcare / Dental Support Org,2
...,...,...
179,B2B Commerce / FinTech,1
180,Fitness / HealthTech,1
181,HealthTech / Fitness,1
182,EdTech / Higher Education,1


In [39]:
def classify_industry(industry):

    if pd.isna(industry):
        return "Unknown"

    x = str(industry).lower().strip()

    # =========================================================
    # 1. EDUCATION
    # =========================================================
    if any(k in x for k in [
        "edtech",
        "education",
        "learning",
        "exam preparation",
        "exam prep",
        "test prep",
        "tutoring",
        "k-12",
        "higher education",
        "professional training",
        "professional learning",
        "career education",
        "career & internship",
        "study abroad",
        "bootcamp",
        "computer education",
        "school partnerships",
        "paramedical training"
    ]):
        return "Education"


    # =========================================================
    # 2. FINANCIAL SERVICES
    # =========================================================
    elif any(k in x for k in [
        "fintech",
        "insurtech",
        "wealthtech",
        "regtech",
        "financial products",
        "financial services",
        "payments",
        "payment",
        "lending",
        "nbfc",
        "banking",
        "banking-as-a-service",
        "banking api",
        "corporate cards",
        "spend management",
        "revenue-based financing",
        "fixed-income",
        "fintech data",
        "fintech api"
    ]):
        return "Financial Services"


    # =========================================================
    # 3. HEALTHCARE
    # =========================================================
    elif any(k in x for k in [
        "healthtech",
        "healthcare",
        "health tech",
        "medical",
        "clinic",
        "hospital",
        "telehealth",
        "dental",
        "diagnostics",
        "digital health",
        "surgery care",
        "epharmacy",
        "pharmacy",
        "metabolic health",
        "therapeutics"
    ]):
        return "Healthcare"


    # =========================================================
    # 4. RECRUITMENT & HR
    # =========================================================
    elif any(k in x for k in [
        "recruitment",
        "staffing",
        "hr tech",
        "hr services",
        "human resources",
        "workforce",
        "hcm",
        "payroll",
        "compensation",
        "gig workforce",
        "career assistance"
    ]):
        return "Recruitment & HR"


    # =========================================================
    # 5. LOGISTICS & SUPPLY CHAIN
    # =========================================================
    elif any(k in x for k in [
        "logistics",
        "last-mile",
        "last mile",
        "supply chain",
        "delivery",
        "freighttech",
        "freight",
        "ndr automation",
        "e-commerce logistics",
        "packaging & supply chain",
        "packaging",
        "logistics saas",
        "HappyLocate"
    ]):
        return "Logistics & Supply Chain"


    # =========================================================
    # 6. REAL ESTATE
    # =========================================================
    elif any(k in x for k in [
        "proptech",
        "real estate",
        "property management",
        "property",
        "co-living",
        "housing",
        "mortgages",
        "multifamily",
        "realty",
        "real estate investment",
        "real estate development"
    ]):
        return "Real Estate"


    # =========================================================
    # 7. AUTOMOTIVE & MOBILITY
    # =========================================================
    elif any(k in x for k in [
        "autotech",
        "automotive",
        "automobile",
        "used cars",
        "car services",
        "car maintenance",
        "bodyshop",
        "chauffeur",
        "roadside assistance",
        "mobility",
        "ride-hailing",
        "ride hailing",
        "parking",
        "electric mobility",
        "ev",
         "rooftop solar"
    ]):
        return "Automotive & Mobility"


    # =========================================================
    # 8. E-COMMERCE & RETAIL
    # =========================================================
    elif any(k in x for k in [
        "e-commerce",
        "ecommerce",
        "quick commerce",
        "retail",
        "d2c",
        "consumer commerce",
        "eyewear",
        "food delivery",
        "marketplace",
        "commerce"
    ]):
        return "E-commerce & Retail"


    # =========================================================
    # 9. MARKETING
    # =========================================================
    elif any(k in x for k in [
        "marketing",
        "martech",
        "advertising",
        "customer engagement",
        "retention marketing",
        "email marketing",
        "loyalty",
        "crm",
        "cro",
        "customer experience"
    ]):
        return "Marketing"


    # =========================================================
    # 10. TRAVEL & HOSPITALITY
    # =========================================================
    elif any(k in x for k in [
        "travel",
        "tourism",
        "hospitality",
        "hotel",
        "vacation rental",
        "airbnb"
    ]):
        return "Travel & Hospitality"


    # =========================================================
    # 11. HOME SERVICES
    # =========================================================
    elif any(k in x for k in [
        "interior design",
        "home renovation",
        "home & furniture",
        "home service",
        "appliance repair"
    ]):
        return "Home Services"


    # =========================================================
    # 12. B2B / INDUSTRIAL
    # =========================================================
    elif any(k in x for k in [
        "b2b commerce",
        "b2b industrial",
        "industrial commerce",
        "manufacturing marketplace",
        "b2b packaging"
    ]):
        return "B2B & Industrial"


    # =========================================================
    # 13. TECHNOLOGY
    # =========================================================
    elif any(k in x for k in [
        "technology",
        "software",
        "saas",
        "ai",
        "analytics",
        "data",
        "cloud",
        "developer infrastructure",
        "automation",
        "conversational",
        "contact center",
        "ccaas",
        "cpaas",
        "telephony",
        "api",
        "digital transformation",
        "workflow",
        "low-code",
        "helpdesk",
        "revops",
        "sales execution",
        "sales engagement",
        "sales automation",
        "digital adoption",
        "subscription billing",
        "contract lifecycle",
        "legal practice",
        "identity verification",
        "compliance automation",
        "data catalog",
        "it managed service",
        "product documentation",
        "business communications",
        "Relocation Tech "
    ]):
        return "Technology"


    # =========================================================
    # 14. OTHER
    # =========================================================
    else:
        return "Other"

In [17]:
clean_df["Industry_Main"] = (
    clean_df["Industry"]
    .apply(classify_industry)
)

In [18]:
clean_df[
    ["Company", "Industry", "Industry_Main"]
].head(20)

,Company,Industry,Industry_Main
0,Staffyn,Recruitment & Staffing,Recruitment & HR
1,Progres InnoTech Pvt Ltd,Recruitment & HR Tech / Software,Recruitment & HR
2,Aarizon Services,Technology Recruitment & Staffing,Recruitment & HR
3,Origin Hiring,IT Recruitment & Staffing,Recruitment & HR
4,CipherSchools,EdTech / Career Education,Education
5,Toddle,EdTech / K-12 SaaS,Education
6,UPRIO,EdTech / AI Learning,Education
7,SuperKalam,EdTech / AI Exam Preparation,Education
8,Kalvium,EdTech / Work-Integrated Higher Education,Education
9,Multibhashi,EdTech / Language Learning,Education


In [45]:
clean_df["Industry_Main"].value_counts()

Industry_Main
Technology                  31
Healthcare                  29
Financial Services          29
Education                   26
Automotive & Mobility       19
Real Estate                 18
Recruitment & HR            13
Logistics & Supply Chain    12
E-commerce & Retail         10
Marketing                   10
Home Services                4
Other                        1
Travel & Hospitality         1
Name: count, dtype: int64

In [46]:
other_df = clean_df[
    clean_df["Industry_Main"] == "Other"
][
    ["Company", "Industry"]
]

other_df

,Company,Industry
42,HappyLocate,Relocation Tech / Packers & Movers Platform


In [48]:
industry_validation = (
    clean_df[
        ["Industry", "Industry_Main"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Industry_Main", "Industry"]
    )
)

industry_validation

,Industry,Industry_Main
61,AI / Software Development,Automotive & Mobility
21,AutoTech / 24/7 Roadside Assistance,Automotive & Mobility
53,AutoTech / Bodyshop & Car Maintenance,Automotive & Mobility
36,AutoTech / On-Demand Chauffeurs,Automotive & Mobility
37,AutoTech / Smart Parking & Car Services,Automotive & Mobility
...,...,...
161,Sales Outsourcing / Telecalling AI,Technology
122,"Salon, Spa & Fitness Booking Software",Technology
93,Subscription Billing SaaS,Technology
102,Workflow / Low-Code Automation,Technology


In [49]:
print("Total companies:", len(clean_df))

Total companies: 203


In [50]:
print(
    "Unique companies:",
    clean_df["Company"].nunique()
)

Unique companies: 203


In [51]:
pd.crosstab(
    clean_df["Industry_Main"],
    clean_df["Industry"]
)

Industry,AI & Analytics Solutions,AI / NLP / Enterprise AI,AI / SaaS / CRM Software,AI / Software / Business Automation,AI / Software / Robotics,AI / Software Development,AI Sales Training / EdTech SaaS,AutoTech / 24/7 Roadside Assistance,AutoTech / Bodyshop & Car Maintenance,AutoTech / On-Demand Chauffeurs,...,Software Development / AI & Automation,Spend Management / Corporate Cards,Staffing / HR Services,Subscription Billing SaaS,Technology Recruitment & Staffing,Telehealth & Healthcare Practice Management,Travel & Tourism,Vacation Rental & Airbnb Property Management,WealthTech / Curated Portfolio Management,Workflow / Low-Code Automation
Industry_Main,,,,,,,,,,,,,,,,,,,,,
Automotive & Mobility,0,0,0,0,0,1,0,1,1,1,...,1,0,0,0,0,0,0,0,0,0
E-commerce & Retail,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Education,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Financial Services,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
Healthcare,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
Home Services,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Logistics & Supply Chain,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Marketing,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Other,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [52]:
clean_df["Industry_Main"] = (
    clean_df["Industry"]
    .apply(classify_industry)
)

In [53]:
clean_df["Industry_Main"].value_counts()

Industry_Main
Technology                  31
Healthcare                  29
Financial Services          29
Education                   26
Automotive & Mobility       19
Real Estate                 18
Recruitment & HR            13
Logistics & Supply Chain    12
E-commerce & Retail         10
Marketing                   10
Home Services                4
Other                        1
Travel & Hospitality         1
Name: count, dtype: int64

In [54]:
other_df = clean_df[
    clean_df["Industry_Main"] == "Other"
][
    ["Company", "Industry"]
]

other_df

,Company,Industry
42,HappyLocate,Relocation Tech / Packers & Movers Platform


In [56]:
clean_df["Industry_Main"] = (
    clean_df["Industry"]
    .apply(classify_industry)
)

In [57]:
clean_df[
    clean_df["Company"].isin([
        "HappyLocate",
        "SolarSquare Energy"
    ])
][
    ["Company", "Industry", "Industry_Main"]
]

,Company,Industry,Industry_Main
42,HappyLocate,Relocation Tech / Packers & Movers Platform,Other
48,SolarSquare Energy,CleanTech / Rooftop Solar SMB,Automotive & Mobility


In [59]:
clean_df[
    clean_df["Industry_Main"] == "Other"
][["Company", "Industry"]]

,Company,Industry
42,HappyLocate,Relocation Tech / Packers & Movers Platform


In [60]:
print("Total companies:", len(clean_df))
print("Unique companies:", clean_df["Company"].nunique())
print("Other companies:", (clean_df["Industry_Main"] == "Other").sum())

Total companies: 203
Unique companies: 203
Other companies: 1


In [61]:
clean_df[
    ["Company", "CEO_Email", "Current_Email"]
].head(20)

,Company,CEO_Email,Current_Email
0,Staffyn,<NA>,hr@staffyn.in
1,Progres InnoTech Pvt Ltd,<NA>,hr@progresinnotech.com
2,Aarizon Services,<NA>,contact@aarizon.com
3,Origin Hiring,<NA>,info@originhiring.com
4,CipherSchools,<NA>,support@cipherschools.com
5,Toddle,<NA>,hello@toddleapp.com; support@toddleapp.com
6,UPRIO,<NA>,developers@uprio.com
7,SuperKalam,<NA>,ask@superkalam.com; hello@superkalam.com
8,Kalvium,<NA>,info@kalvium.com
9,Multibhashi,<NA>,support@multibhashi.com


In [62]:
print("CEO Email missing:", clean_df["CEO_Email"].isna().sum())
print("Current Email missing:", clean_df["Current_Email"].isna().sum())

CEO Email missing: 191
Current Email missing: 30


In [65]:
def extract_email(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    # Find email addresses inside the text
    emails = re.findall(
        r'[\w\.-]+@[\w\.-]+\.\w+',
        value
    )

    if not emails:
        return np.nan

    # Return the first email found
    return emails[0]

In [66]:
clean_df["CEO_Email_Clean"] = (
    clean_df["CEO_Email"]
    .apply(extract_email)
)

clean_df["Current_Email_Clean"] = (
    clean_df["Current_Email"]
    .apply(extract_email)
)

In [67]:
clean_df[
    [
        "Company",
        "CEO_Email",
        "CEO_Email_Clean",
        "Current_Email",
        "Current_Email_Clean"
    ]
].head(20)

,Company,CEO_Email,CEO_Email_Clean,Current_Email,Current_Email_Clean
0,Staffyn,<NA>,NaN,hr@staffyn.in,hr@staffyn.in
1,Progres InnoTech Pvt Ltd,<NA>,NaN,hr@progresinnotech.com,hr@progresinnotech.com
2,Aarizon Services,<NA>,NaN,contact@aarizon.com,contact@aarizon.com
3,Origin Hiring,<NA>,NaN,info@originhiring.com,info@originhiring.com
4,CipherSchools,<NA>,NaN,support@cipherschools.com,support@cipherschools.com
5,Toddle,<NA>,NaN,hello@toddleapp.com; support@toddleapp.com,hello@toddleapp.com
6,UPRIO,<NA>,NaN,developers@uprio.com,developers@uprio.com
7,SuperKalam,<NA>,NaN,ask@superkalam.com; hello@superkalam.com,ask@superkalam.com
8,Kalvium,<NA>,NaN,info@kalvium.com,info@kalvium.com
9,Multibhashi,<NA>,NaN,support@multibhashi.com,support@multibhashi.com


In [68]:
def is_valid_email(value):

    if pd.isna(value):
        return 0

    email = str(value).strip().lower()

    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

    return int(bool(re.match(pattern, email)))

In [69]:
clean_df["CEO_Email_Valid"] = (
    clean_df["CEO_Email_Clean"]
    .apply(is_valid_email)
)

clean_df["Current_Email_Valid"] = (
    clean_df["Current_Email_Clean"]
    .apply(is_valid_email)
)

In [70]:
print(
    "Valid CEO emails:",
    clean_df["CEO_Email_Valid"].sum()
)

print(
    "Valid Current emails:",
    clean_df["Current_Email_Valid"].sum()
)

Valid CEO emails: 12
Valid Current emails: 170


In [71]:
print(
    "Invalid CEO emails:",
    (clean_df["CEO_Email_Valid"] == 0).sum()
)

print(
    "Invalid Current emails:",
    (clean_df["Current_Email_Valid"] == 0).sum()
)

Invalid CEO emails: 191
Invalid Current emails: 33


In [72]:
invalid_current = clean_df[
    clean_df["Current_Email"].notna() &
    (clean_df["Current_Email_Valid"] == 0)
][
    ["Company", "Current_Email"]
]

invalid_current

,Company,Current_Email
92,Whatfix,Not Published
94,Zluri,Not Published
120,DoorLoop,https://doorloop.notion.site/1dceb00b7a0b81909...


In [73]:
invalid_ceo = clean_df[
    clean_df["CEO_Email"].notna() &
    (clean_df["CEO_Email_Valid"] == 0)
][
    ["Company", "CEO_Email"]
]

invalid_ceo

,Company,CEO_Email


In [74]:
clean_df["Email_Available"] = (
    (
        (clean_df["Current_Email_Valid"] == 1) |
        (clean_df["CEO_Email_Valid"] == 1)
    )
    .astype(int)
)

In [75]:
clean_df["Email_Available"].value_counts()

Email_Available
1    170
0     33
Name: count, dtype: int64

In [76]:
email_coverage = (
    clean_df["Email_Available"].mean() * 100
)

print(f"Email Coverage: {email_coverage:.1f}%")

Email Coverage: 83.7%


In [77]:
clean_df["Missing_Email"] = (
    clean_df["Email_Available"] == 0
).astype(int)

In [78]:
print(
    "Companies missing usable email:",
    clean_df["Missing_Email"].sum()
)

Companies missing usable email: 33


In [79]:
def email_type(row):

    current = row["Current_Email_Clean"]
    ceo = row["CEO_Email_Clean"]

    if pd.notna(ceo):
        return "Leadership Email"

    if pd.notna(current):
        email = str(current).lower()

        generic_words = [
            "info@",
            "hello@",
            "contact@",
            "support@",
            "sales@",
            "admin@",
            "enquiry@",
            "enquiries@",
            "marketing@",
            "business@",
            "care@"
        ]

        if any(word in email for word in generic_words):
            return "Generic Business Email"

        return "Business Email"

    return "No Email"

In [80]:
clean_df["Email_Type"] = (
    clean_df.apply(email_type, axis=1)
)

In [81]:
clean_df["Email_Type"].value_counts()

Email_Type
Generic Business Email    108
Business Email             50
No Email                   33
Leadership Email           12
Name: count, dtype: int64

In [82]:
def email_priority(row):

    if row["Email_Type"] == "Leadership Email":
        return "High"

    elif row["Email_Type"] == "Business Email":
        return "Medium"

    elif row["Email_Type"] == "Generic Business Email":
        return "Low"

    else:
        return "Missing"

In [83]:
clean_df["Email_Priority"] = (
    clean_df.apply(email_priority, axis=1)
)

In [84]:
clean_df[
    [
        "Company",
        "CEO_Email",
        "CEO_Email_Clean",
        "Current_Email",
        "Current_Email_Clean",
        "Email_Available",
        "Email_Type",
        "Email_Priority"
    ]
].head(30)

,Company,CEO_Email,CEO_Email_Clean,Current_Email,Current_Email_Clean,Email_Available,Email_Type,Email_Priority
0,Staffyn,<NA>,NaN,hr@staffyn.in,hr@staffyn.in,1,Business Email,Medium
1,Progres InnoTech Pvt Ltd,<NA>,NaN,hr@progresinnotech.com,hr@progresinnotech.com,1,Business Email,Medium
2,Aarizon Services,<NA>,NaN,contact@aarizon.com,contact@aarizon.com,1,Generic Business Email,Low
3,Origin Hiring,<NA>,NaN,info@originhiring.com,info@originhiring.com,1,Generic Business Email,Low
4,CipherSchools,<NA>,NaN,support@cipherschools.com,support@cipherschools.com,1,Generic Business Email,Low
5,Toddle,<NA>,NaN,hello@toddleapp.com; support@toddleapp.com,hello@toddleapp.com,1,Generic Business Email,Low
6,UPRIO,<NA>,NaN,developers@uprio.com,developers@uprio.com,1,Business Email,Medium
7,SuperKalam,<NA>,NaN,ask@superkalam.com; hello@superkalam.com,ask@superkalam.com,1,Business Email,Medium
8,Kalvium,<NA>,NaN,info@kalvium.com,info@kalvium.com,1,Generic Business Email,Low
9,Multibhashi,<NA>,NaN,support@multibhashi.com,support@multibhashi.com,1,Generic Business Email,Low


In [85]:
clean_df[["Company", "Phone"]].head(30)

,Company,Phone
0,Staffyn,+91 88752 84059
1,Progres InnoTech Pvt Ltd,+91 78299 16411
2,Aarizon Services,+91 70562 19573
3,Origin Hiring,<NA>
4,CipherSchools,<NA>
5,Toddle,+91 98 2525 7365
6,UPRIO,+91 7624935001
7,SuperKalam,+91 9319720944
8,Kalvium,+91 9483 200 300
9,Multibhashi,+91 95356 85555


In [86]:
print("Missing phone numbers:", clean_df["Phone"].isna().sum())

Missing phone numbers: 55


In [87]:
def clean_phone_number(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Keep only digits
    digits = re.sub(r"\D", "", value)

    # If there are no digits
    if not digits:
        return np.nan

    # Handle Indian country code
    if digits.startswith("91") and len(digits) == 12:
        digits = digits[2:]

    # Valid Indian mobile number should normally be 10 digits
    if len(digits) == 10 and digits[0] in "6789":
        return digits

    return np.nan

In [88]:
clean_df["Phone_Clean"] = (
    clean_df["Phone"]
    .apply(clean_phone_number)
)

In [89]:
clean_df[
    ["Company", "Phone", "Phone_Clean"]
].head(30)

,Company,Phone,Phone_Clean
0,Staffyn,+91 88752 84059,8875284059
1,Progres InnoTech Pvt Ltd,+91 78299 16411,7829916411
2,Aarizon Services,+91 70562 19573,7056219573
3,Origin Hiring,<NA>,NaN
4,CipherSchools,<NA>,NaN
5,Toddle,+91 98 2525 7365,9825257365
6,UPRIO,+91 7624935001,7624935001
7,SuperKalam,+91 9319720944,9319720944
8,Kalvium,+91 9483 200 300,9483200300
9,Multibhashi,+91 95356 85555,9535685555


In [90]:
clean_df["Phone_Valid"] = (
    clean_df["Phone_Clean"].notna().astype(int)
)

In [91]:
clean_df["Phone_Valid"].value_counts()

Phone_Valid
0    123
1     80
Name: count, dtype: int64

In [92]:
clean_df["Phone_Available"] = (
    clean_df["Phone_Valid"]
)

In [93]:
print(
    "Companies with valid phone:",
    clean_df["Phone_Available"].sum()
)

print(
    "Companies without valid phone:",
    (clean_df["Phone_Available"] == 0).sum()
)

Companies with valid phone: 80
Companies without valid phone: 123


In [94]:
phone_coverage = (
    clean_df["Phone_Available"].mean() * 100
)

print(f"Phone Coverage: {phone_coverage:.1f}%")

Phone Coverage: 39.4%


In [95]:
invalid_phones = clean_df[
    clean_df["Phone"].notna() &
    clean_df["Phone_Clean"].isna()
][
    ["Company", "Phone"]
]

invalid_phones

,Company,Phone
12,Porter,022-4410-4410
22,Dezerv,022-48930368
23,Shipway,+91 124 427 9900
26,Toddle (Teacher App),+1 561-985-5147
32,HomeLane,1800-102-4663
...,...,...
188,Swiggy,08068422422
197,HealthifyMe,18004199501
199,Vedantu,1800-120-456-456
200,upGrad,1800-210-2020


In [96]:
clean_df["Missing_Phone"] = (
    clean_df["Phone_Available"] == 0
).astype(int)

In [97]:
print(
    "Missing/invalid phone numbers:",
    clean_df["Missing_Phone"].sum()
)

Missing/invalid phone numbers: 123


In [98]:
def phone_quality(value):

    if pd.isna(value):
        return "Missing"

    value = str(value)

    if len(value) == 10 and value[0] in "6789":
        return "Valid"

    return "Invalid"

In [99]:
clean_df["Phone_Quality"] = (
    clean_df["Phone_Clean"]
    .apply(phone_quality)
)

In [100]:
clean_df["Phone_Quality"].value_counts()

Phone_Quality
Missing    123
Valid       80
Name: count, dtype: int64

In [101]:
clean_df[
    [
        "Company",
        "Phone",
        "Phone_Clean",
        "Phone_Valid",
        "Phone_Available",
        "Phone_Quality",
        "Missing_Phone"
    ]
].head(30)

,Company,Phone,Phone_Clean,Phone_Valid,Phone_Available,Phone_Quality,Missing_Phone
0,Staffyn,+91 88752 84059,8875284059,1,1,Valid,0
1,Progres InnoTech Pvt Ltd,+91 78299 16411,7829916411,1,1,Valid,0
2,Aarizon Services,+91 70562 19573,7056219573,1,1,Valid,0
3,Origin Hiring,<NA>,NaN,0,0,Missing,1
4,CipherSchools,<NA>,NaN,0,0,Missing,1
5,Toddle,+91 98 2525 7365,9825257365,1,1,Valid,0
6,UPRIO,+91 7624935001,7624935001,1,1,Valid,0
7,SuperKalam,+91 9319720944,9319720944,1,1,Valid,0
8,Kalvium,+91 9483 200 300,9483200300,1,1,Valid,0
9,Multibhashi,+91 95356 85555,9535685555,1,1,Valid,0


In [102]:
clean_df[["Company", "Website"]].head(20)

,Company,Website
0,Staffyn,https://www.staffyn.in/
1,Progres InnoTech Pvt Ltd,https://progresinnotech.com/
2,Aarizon Services,https://www.aarizon.com/
3,Origin Hiring,https://www.originhiring.com/
4,CipherSchools,https://www.cipherschools.com/
5,Toddle,https://www.toddleapp.com/
6,UPRIO,https://www.uprio.com/
7,SuperKalam,https://superkalam.com/
8,Kalvium,https://kalvium.com/
9,Multibhashi,https://multibhashi.com/


In [103]:
def clean_website(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value == "":
        return np.nan

    # Remove spaces
    value = value.replace(" ", "")

    # Add protocol if missing
    if not value.startswith(("http://", "https://")):
        value = "https://" + value

    return value

In [104]:
clean_df["Website_Clean"] = (
    clean_df["Website"]
    .apply(clean_website)
)

In [105]:
clean_df["Website_Available"] = (
    clean_df["Website_Clean"].notna().astype(int)
)

In [106]:
def is_valid_website(value):

    if pd.isna(value):
        return 0

    value = str(value).strip().lower()

    pattern = r"^https?://[^\s]+\.[^\s]+$"

    return int(bool(re.match(pattern, value)))

In [107]:
clean_df["Website_Valid"] = (
    clean_df["Website_Clean"]
    .apply(is_valid_website)
)

In [108]:
clean_df["Missing_Website"] = (
    clean_df["Website_Available"] == 0
).astype(int)

In [109]:
website_coverage = (
    clean_df["Website_Available"].mean() * 100
)

print(f"Website Coverage: {website_coverage:.1f}%")

Website Coverage: 95.1%


In [110]:
invalid_websites = clean_df[
    clean_df["Website"].notna() &
    (clean_df["Website_Valid"] == 0)
][
    ["Company", "Website", "Website_Clean"]
]

invalid_websites

,Company,Website,Website_Clean


In [111]:
clean_df["Leadership_Available"] = (
    clean_df["Leadership"].notna().astype(int)
)

In [112]:
clean_df["Missing_Leadership"] = (
    clean_df["Leadership_Available"] == 0
).astype(int)

In [113]:
leadership_coverage = (
    clean_df["Leadership_Available"].mean() * 100
)

print(f"Leadership Coverage: {leadership_coverage:.1f}%")

Leadership Coverage: 89.7%


In [114]:
leadership_coverage = (
    clean_df["Leadership_Available"].mean() * 100
)

print(f"Leadership Coverage: {leadership_coverage:.1f}%")

Leadership Coverage: 89.7%


In [115]:
clean_df[
    clean_df["Leadership_Available"] == 0
][
    ["Company", "Leadership", "CEO_Email"]
]

,Company,Leadership,CEO_Email
0,Staffyn,<NA>,<NA>
1,Progres InnoTech Pvt Ltd,<NA>,<NA>
2,Aarizon Services,<NA>,<NA>
3,Origin Hiring,<NA>,<NA>
6,UPRIO,<NA>,<NA>
7,SuperKalam,<NA>,<NA>
8,Kalvium,<NA>,<NA>
14,Strata,<NA>,<NA>
15,Sri Sai Ventures,<NA>,<NA>
16,AI2Career,<NA>,<NA>


In [116]:
clean_df["Industry_Available"] = (
    clean_df["Industry_Main"].notna() &
    (clean_df["Industry_Main"] != "Unknown")
).astype(int)

In [117]:
clean_df["Data_Completeness_Score"] = (
    clean_df["Email_Available"] * 30 +
    clean_df["Phone_Available"] * 20 +
    clean_df["Leadership_Available"] * 20 +
    clean_df["Website_Available"] * 20 +
    clean_df["Industry_Available"] * 10
)

In [118]:
def quality_category(score):

    if score >= 80:
        return "High"

    elif score >= 50:
        return "Medium"

    else:
        return "Low"

In [119]:
clean_df["Data_Quality"] = (
    clean_df["Data_Completeness_Score"]
    .apply(quality_category)
)

In [120]:
data_gap_columns = [
    "Missing_Email",
    "Missing_Phone",
    "Missing_Leadership",
    "Missing_Website"
]

clean_df["Data_Gaps"] = (
    clean_df[data_gap_columns]
    .sum(axis=1)
)

In [121]:
def gap_category(gaps):

    if gaps == 0:
        return "Complete"

    elif gaps == 1:
        return "1 Data Gap"

    elif gaps == 2:
        return "2 Data Gaps"

    else:
        return "3+ Data Gaps"

In [122]:
clean_df["Data_Gap_Category"] = (
    clean_df["Data_Gaps"]
    .apply(gap_category)
)

In [123]:
clean_df["Outreach_Score"] = (
    clean_df["Email_Available"] * 40 +
    clean_df["Phone_Available"] * 25 +
    clean_df["Leadership_Available"] * 20 +
    clean_df["Website_Available"] * 15
)

In [124]:
def outreach_priority(score):

    if score >= 80:
        return "High Priority"

    elif score >= 50:
        return "Medium Priority"

    else:
        return "Low Priority"

In [125]:
clean_df["Outreach_Priority"] = (
    clean_df["Outreach_Score"]
    .apply(outreach_priority)
)

In [126]:
def contactability_status(row):

    if row["Email_Available"] == 1 and row["Phone_Available"] == 1:
        return "Email + Phone"

    elif row["Email_Available"] == 1:
        return "Email Only"

    elif row["Phone_Available"] == 1:
        return "Phone Only"

    else:
        return "No Direct Contact"

In [127]:
clean_df["Contactability_Status"] = (
    clean_df.apply(contactability_status, axis=1)
)

In [128]:
clean_df["Data_Quality_Flag"] = np.select(
    [
        clean_df["Data_Completeness_Score"] >= 80,
        clean_df["Data_Completeness_Score"] >= 50
    ],
    [
        "Good",
        "Needs Enrichment"
    ],
    default="Poor"
)

In [129]:
print("=" * 50)
print("FINAL DATA VALIDATION")
print("=" * 50)

print("Total rows:", len(clean_df))
print("Unique companies:", clean_df["Company"].nunique())

print(
    "Duplicate companies:",
    clean_df["Company"].duplicated().sum()
)

print(
    "Industry categories:",
    clean_df["Industry_Main"].nunique()
)

print(
    "Missing emails:",
    clean_df["Missing_Email"].sum()
)

print(
    "Missing phones:",
    clean_df["Missing_Phone"].sum()
)

print(
    "Missing leadership:",
    clean_df["Missing_Leadership"].sum()
)

print(
    "Missing websites:",
    clean_df["Missing_Website"].sum()
)

print("=" * 50)

FINAL DATA VALIDATION
Total rows: 203
Unique companies: 203
Duplicate companies: 0
Industry categories: 13
Missing emails: 33
Missing phones: 123
Missing leadership: 21
Missing websites: 10


In [130]:
quality_summary = (
    clean_df["Data_Quality"]
    .value_counts()
    .reset_index()
)

quality_summary.columns = [
    "Data_Quality",
    "Company_Count"
]

quality_summary

,Data_Quality,Company_Count
0,High,158
1,Medium,43
2,Low,2


In [131]:
industry_summary = (
    clean_df["Industry_Main"]
    .value_counts()
    .reset_index()
)

industry_summary.columns = [
    "Industry_Main",
    "Company_Count"
]

industry_summary

,Industry_Main,Company_Count
0,Technology,31
1,Healthcare,29
2,Financial Services,29
3,Education,26
4,Automotive & Mobility,19
5,Real Estate,18
6,Recruitment & HR,13
7,Logistics & Supply Chain,12
8,E-commerce & Retail,10
9,Marketing,10


In [132]:
contactability_summary = (
    clean_df["Contactability_Status"]
    .value_counts()
    .reset_index()
)

contactability_summary.columns = [
    "Contactability_Status",
    "Company_Count"
]

contactability_summary

,Contactability_Status,Company_Count
0,Email Only,95
1,Email + Phone,75
2,No Direct Contact,28
3,Phone Only,5


In [133]:
data_gap_summary = pd.DataFrame({
    "Data_Field": [
        "Email",
        "Phone",
        "Leadership",
        "Website"
    ],
    "Missing_Count": [
        clean_df["Missing_Email"].sum(),
        clean_df["Missing_Phone"].sum(),
        clean_df["Missing_Leadership"].sum(),
        clean_df["Missing_Website"].sum()
    ]
})

data_gap_summary

,Data_Field,Missing_Count
0,Email,33
1,Phone,123
2,Leadership,21
3,Website,10


In [134]:
final_columns = [
    "Company",
    "Industry",
    "Industry_Main",
    "Leadership",
    "CEO_Email",
    "CEO_Email_Clean",
    "Current_Email",
    "Current_Email_Clean",
    "Email_Available",
    "CEO_Email_Valid",
    "Current_Email_Valid",
    "Email_Type",
    "Email_Priority",
    "Phone",
    "Phone_Clean",
    "Phone_Valid",
    "Phone_Available",
    "Phone_Quality",
    "Website",
    "Website_Clean",
    "Website_Valid",
    "Website_Available",
    "Leadership_Available",
    "Industry_Available",
    "Missing_Email",
    "Missing_Phone",
    "Missing_Leadership",
    "Missing_Website",
    "Data_Gaps",
    "Data_Gap_Category",
    "Data_Completeness_Score",
    "Data_Quality",
    "Outreach_Score",
    "Outreach_Priority",
    "Contactability_Status",
    "Data_Quality_Flag"
]

final_df = clean_df[final_columns].copy()

final_df.head()

,Company,Industry,Industry_Main,Leadership,CEO_Email,CEO_Email_Clean,Current_Email,Current_Email_Clean,Email_Available,CEO_Email_Valid,...,Missing_Leadership,Missing_Website,Data_Gaps,Data_Gap_Category,Data_Completeness_Score,Data_Quality,Outreach_Score,Outreach_Priority,Contactability_Status,Data_Quality_Flag
0,Staffyn,Recruitment & Staffing,Recruitment & HR,<NA>,<NA>,NaN,hr@staffyn.in,hr@staffyn.in,1,0,...,1,0,1,1 Data Gap,80,High,80,High Priority,Email + Phone,Good
1,Progres InnoTech Pvt Ltd,Recruitment & HR Tech / Software,Recruitment & HR,<NA>,<NA>,NaN,hr@progresinnotech.com,hr@progresinnotech.com,1,0,...,1,0,1,1 Data Gap,80,High,80,High Priority,Email + Phone,Good
2,Aarizon Services,Technology Recruitment & Staffing,Recruitment & HR,<NA>,<NA>,NaN,contact@aarizon.com,contact@aarizon.com,1,0,...,1,0,1,1 Data Gap,80,High,80,High Priority,Email + Phone,Good
3,Origin Hiring,IT Recruitment & Staffing,Recruitment & HR,<NA>,<NA>,NaN,info@originhiring.com,info@originhiring.com,1,0,...,1,0,2,2 Data Gaps,60,Medium,55,Medium Priority,Email Only,Needs Enrichment
4,CipherSchools,EdTech / Career Education,Education,Anurag Mishra — Founder,<NA>,NaN,support@cipherschools.com,support@cipherschools.com,1,0,...,0,0,1,1 Data Gap,80,High,75,Medium Priority,Email Only,Good


In [135]:
output_file = "Cleaned_203_Companies.xlsx"

final_df.to_excel(
    output_file,
    index=False,
    sheet_name="Cleaned Companies"
)

print(f"Successfully saved: {output_file}")

Successfully saved: Cleaned_203_Companies.xlsx
